# Proyek Analisis Data: Bike Sharing
- **Nama:** Alvis Aditya
- **Email:** alvisdty@gmail.com
- **ID Dicoding:** alvis_diy

## Menentukan Pertanyaan Bisnis

- Pertanyaan 1: Bagaimana pola perbedaan penyewaan sepeda antara hari kerja (workingday) dan hari libur (holiday) berdasarkan jam (hr), dan pada jam berapa titik puncak permintaan terjadi?
- Pertanyaan 2:Bagaimana perubahan kondisi cuaca dan suhu memengaruhi perbedaan perilaku penyewaan antara pengguna casual dan pengguna registered?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style seaborn
sns.set(style='darkgrid')

## Data Wrangling

### Gathering Data

In [ ]:
# Memuat data day.csv dan hour.csv
day_df = pd.read_csv("data/day.csv")
hour_df = pd.read_csv("data/hour.csv")


In [ ]:

# Menampilkan 5 baris pertama untuk memastikan data masuk
day_df.head()


In [ ]:
hour_df.head()

In [ ]:
hour_df.describe()

In [ ]:
day_df.describe()

**Insight:**
- Data day.csv memuat agregasi harian peminjaman sepeda (731 baris).

- Data hour.csv memuat peminjaman sepeda per jam (17.379 baris), yang sangat berguna untuk menjawab Pertanyaan 1.

### Assessing Data

In [ ]:
# Memeriksa tipe data dan missing values pada hour_df
print("Info hour_df:")
hour_df.info()

In [ ]:
print("\nJumlah duplikasi hour_df: ", hour_df.duplicated().sum())
print("\nJumlah duplikasi day_df: ", day_df.duplicated().sum())

In [ ]:
# Memeriksa tipe data dan missing values pada day_df
print("\nInfo day_df:")
day_df.info()

In [ ]:
day_df.isna().sum()


In [ ]:
hour_df.isna().sum()

**Insight:**
- Berdasarkan hasil inspeksi di sistem saya, dataset ini sangat bersih. Tidak ada missing values (Nulls = 0) pada kedua dataset, dan tidak ada baris yang terduplikasi.

- Namun, tipe data pada kolom dteday masih berupa object (string), padahal merepresentasikan tanggal. Selain itu, beberapa kolom kategorik seperti season, weathersit, weekday direpresentasikan sebagai int64.

### Cleaning Data

In [ ]:
day_df.drop_duplicates(inplace=True)
hour_df.drop_duplicates(inplace=True)
print("Jumlah duplikasi day_df: ", day_df.duplicated().sum())
print("Jumlah duplikasi hour_df: ", hour_df.duplicated().sum())

In [ ]:
# Mengubah tipe data dteday menjadi datetime
day_df['dteday'] = pd.to_datetime(day_df['dteday'])
hour_df['dteday'] = pd.to_datetime(hour_df['dteday'])

In [ ]:
# Memastikan perubahan tipe data
print(day_df.dtypes['dteday'])
print(hour_df.dtypes['dteday'])

**Insight:**
- Kolom tanggal (dteday) berhasil dikonversi ke format yang benar (datetime), yang akan memudahkan agregasi time-series ke depannya.


## Exploratory Data Analysis (EDA)

### Explore ...

In [ ]:
# Eksplorasi Pertanyaan 1: Agregasi per jam berdasarkan hari kerja vs hari libur
rental_jam = hour_df.groupby(by=['hr', 'workingday']).agg({
    'cnt': 'mean'
}).reset_index()

In [ ]:
rental_jam.describe()

**Insight:**

- Dari agregasi per jam, terlihat rata-rata penyewaan tertinggi terkonsentrasi pada jam-jam tertentu yang membedakan pola hari kerja dan hari libur. Pola ini akan lebih jelas jika divisualisasikan menggunakan Line Chart di tahap selanjutnya.

In [ ]:
# Eksplorasi: Pengaruh Musim Terhadap Jumlah Penyewaan
# Mapping angka musim menjadi teks agar mudah dipahami
season_mapping = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
day_df['season_label'] = day_df['season'].map(season_mapping)

season_analysis = day_df.groupby('season_label')['cnt'].sum().reset_index()

# Visualisasi Pie Chart untuk Musim
plt.figure(figsize=(8, 5))
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']
plt.pie(
    season_analysis['cnt'], 
    labels=season_analysis['season_label'], 
    autopct='%1.1f%%', 
    colors=colors, 
    startangle=90, 
    wedgeprops={'edgecolor': 'white'}
)
plt.title('Proporsi Total Penyewaan Sepeda Berdasarkan Musim', fontsize=14, fontweight='bold')
plt.show()

**Insight:**
- Secara proporsi, musim Gugur (Fall) dan Panas (Summer) adalah peak season yang menyumbang peminjaman tertinggi.

- Artinya, suhu lingkungan yang hangat/sejuk sangat mendukung bisnis ini, sedangkan musim dingin menekan demand penyewaan secara agregat.

In [ ]:
# Eksplorasi Pertanyaan 2: Melihat pengaruh cuaca
eksplorasi_cuaca = day_df.groupby('weathersit')[['casual', 'registered']].mean().reset_index()
display(eksplorasi_cuaca)

**Insight:**
- Perbedaan paling signifikan terlihat pada pengguna casual yang sangat sensitif terhadap perubahan cuaca. Ketika kondisi cuaca memburuk, jumlah penyewaan dari pengguna casual menurun secara drastis. Sebaliknya, pengguna registered masih menunjukkan tingkat penggunaan yang relatif stabil karena kemungkinan besar menggunakan sepeda untuk mobilitas rutin.

In [ ]:
# Eksplorasi: Dinamika Penyewaan Harian
daily_rentals = day_df.groupby('dteday').agg({'cnt': 'sum'}).reset_index()

# Visualisasi Tren Penyewaan Harian
plt.figure(figsize=(16, 6))
plt.plot(
    daily_rentals["dteday"],
    daily_rentals["cnt"],
    linewidth=1.5,
    color="#2E86C1",
    alpha=0.9
)
plt.title('Dinamika Penyewaan Harian (2011-2012)', fontsize=16, fontweight='bold')
plt.xlabel('Tanggal', fontsize=12)
plt.ylabel('Total Penyewaan', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Insight:**

- Tren penyewaan sepeda menunjukkan peningkatan yang signifikan dari tahun 2011 ke 2012.

- Terdapat fluktuasi yang jelas di mana penyewaan selalu memuncak di pertengahan tahun (musim panas/gugur) dan anjlok di akhir/awal tahun (musim dingin).

## Visualization & Explanatory Analysis

### Pertanyaan 1:

In [ ]:
plt.figure(figsize=(12, 6))

# Membuat line plot
sns.lineplot(
    x='hr', 
    y='cnt', 
    hue='workingday', 
    data=rental_jam, 
    palette=['#FF9999', '#66B2FF'], 
    linewidth=2.5
)

# Kustomisasi plot
plt.title('Pola Penyewaan Sepeda per Jam: Hari Kerja vs Akhir Pekan/Libur', fontsize=16, fontweight='bold')
plt.xlabel('Jam (0-23)', fontsize=12)
plt.ylabel('Rata-rata Jumlah Sewa (cnt)', fontsize=12)
plt.xticks(np.arange(0, 24, 1))
plt.legend(title='Hari Kerja (1=Ya, 0=Tidak)', loc='upper left')
plt.tight_layout()
plt.show()

**Insight:**
- Terdapat perbedaan yang sangat jelas antara hari kerja dan hari libur. Saat hari kerja lonjakan permintaan terjadi secara tajam di dua waktu tertentu yaitu jam 08.00 pagi dan 17.00 sore
- Sebaliknya pada hari libur(workingday=0), kurva memuncak secara perlahan di siang hari 

### Pertanyaan 2:

In [ ]:
# Visualisasi Pertanyaan 2: Bar Chart Cuaca (Casual vs Registered)

# 1. Mapping angka cuaca menjadi teks yang mudah dipahami stakeholder
def cluster_weather(weathersit):
    if weathersit == 1:
        return 'Clear'
    elif weathersit == 2:
        return 'Misty'
    else: # Menggabungkan weathersit 3 dan 4 menjadi satu kategori ekstrem
        return 'Bad Weather'

day_df['weather_cluster'] = day_df['weathersit'].apply(cluster_weather)

# 2. Groupby data yang sudah diklaster
weather_clustering = day_df.groupby('weather_cluster')[['casual', 'registered']].mean().reset_index()

# 3. Melt untuk plot berdampingan (Casual vs Registered)
melted_weather = weather_clustering.melt(
    id_vars='weather_cluster', 
    value_vars=['casual', 'registered'], 
    var_name='user_type', 
    value_name='average_rentals'
)

# 4. Eksekusi Visualisasi
plt.figure(figsize=(10, 6))
sns.barplot(
    x='weather_cluster', 
    y='average_rentals', 
    hue='user_type', 
    data=melted_weather, 
    palette='viridis', 
    order=['Clear', 'Misty', 'Bad Weather']
)

plt.title('Dampak Cuaca Terhadap Tipe Pengguna (Casual vs Registered)', fontsize=16, fontweight='bold')
plt.xlabel('Kondisi Cuaca', fontsize=12)
plt.ylabel('Rata-rata Penyewaan', fontsize=12)
plt.tight_layout()
plt.show()

**Insight:**

- Dari grafik batang di atas, terlihat sangat jelas bahwa cuaca adalah deal-breaker bagi penyewa sepeda.

- Saat cuaca 'Clear' (Cerah), penyewaan dari pengguna Casual maupun Registered sangat maksimal.

- Namun, saat cuaca berubah menjadi 'Bad Weather' (Hujan/Badai), pengguna Casual (oranye) anjlok drastis nyaris menyentuh angka nol, menunjukkan mereka sangat sensitif terhadap cuaca. Di sisi lain, pengguna Registered (hijau) masih mempertahankan jumlah sewa yang lumayan untuk mobilitas wajib.

## Analisis Lanjutan (Opsional)

In [ ]:
# ANALISIS LANJUTAN: Pengaruh Kombinasi Cuaca dan Suhu terhadap Total Penyewaan (cnt)

# 1. Pastikan kolom temp_cluster (Cold, Mild, Hot) sudah ada
def categorize_temp(temp_norm):
    if temp_norm < 0.35:
        return 'Cold'
    elif temp_norm < 0.65:
        return 'Mild'
    else:
        return 'Hot'
        
day_df['temp_cluster'] = day_df['temp'].apply(categorize_temp)

# 2. Pastikan kolom weather_cluster (Clear, Misty, Bad Weather) sudah ada
def cluster_weather(weathersit):
    if weathersit == 1:
        return 'Clear'
    elif weathersit == 2:
        return 'Misty'
    else:
        return 'Bad Weather'
        
day_df['weather_cluster'] = day_df['weathersit'].apply(cluster_weather)

# 3. Membuat Pivot Table untuk Heatmap
pivot_weather_temp = day_df.pivot_table(
    index='weather_cluster', 
    columns='temp_cluster', 
    values='cnt', 
    aggfunc='mean'
)

# Susun urutan agar logis di grafik
pivot_weather_temp = pivot_weather_temp[['Cold', 'Mild', 'Hot']]
pivot_weather_temp = pivot_weather_temp.reindex(['Clear', 'Misty', 'Bad Weather'])

# 4. Visualisasi dengan Heatmap Seaborn
plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot_weather_temp, 
    annot=True, 
    fmt=".0f", 
    cmap="YlGnBu",
    linewidths=.5,
    cbar_kws={'label': 'Rata-rata Penyewaan (cnt)'}
)

plt.title('Heatmap: Interaksi Cuaca & Suhu Terhadap Penyewaan Sepeda', fontsize=16, fontweight='bold')
plt.xlabel('Klaster Suhu', fontsize=12)
plt.ylabel('Klaster Cuaca (Kondisi Langit)', fontsize=12)
plt.tight_layout()
plt.show()

# Simpan data yang sudah punya fitur lengkap untuk Dashboard
day_df.to_csv("main_data.csv", index=False)

**Insight:**
- Dari Heatmap di atas, terbukti bahwa Cuaca dan Suhu saling bekerja sama dalam memengaruhi keputusan pelanggan.

- Kondisi Paling Optimal (Sweet Spot): Rata-rata penyewaan meledak sangat tinggi saat langit Cerah (Clear) dan suhu hangat (Hot).

- Saat suhu dingin (Cold), pelanggan masih lumayan banyak yang menyewa ASALKAN langitnya Cerah (Clear). Namun, jika cuaca sudah buruk (Bad Weather), entah suhunya Cold atau Mild, penyewaan tetap hancur lebur (angka terendah).

- Kesimpulan: Cuaca (Kondisi Langit) adalah faktor veto (pembatalan mutlak), sedangkan Suhu (Suhu di Kulit) adalah faktor pendorong (pengali jumlah sewa).

## Conclusion

- Conclusion pertanyaan 1: Pola penyewaan sepeda menunjukkan perbedaan perilaku yang sangat jelas antara hari kerja dan hari libur. Pada hari kerja (workingday=1), permintaan meningkat tajam pada dua waktu utama yaitu sekitar pukul 08.00 pagi dan 17.00 sore. Pola ini mengindikasikan bahwa sepeda banyak digunakan sebagai sarana transportasi komuter untuk berangkat dan pulang kerja. Sebaliknya, pada akhir pekan atau hari libur (workingday=0), pola penyewaan lebih stabil dengan puncak yang terjadi pada siang hari sekitar pukul 12.00 hingga 15.00, yang menunjukkan bahwa penggunaan sepeda cenderung bersifat rekreasi atau aktivitas santai.

    Temuan ini menunjukkan bahwa waktu operasional memiliki pengaruh kuat terhadap pola permintaan. Oleh karena itu, pengelola layanan sepeda dapat mengoptimalkan distribusi armada dengan menempatkan lebih banyak sepeda di area perkantoran dan stasiun transportasi pada jam komuter di hari kerja, serta di area rekreasi atau taman kota pada siang hari di akhir pekan.
- Conclusion pertanyaan 2: Interaksi antara kondisi cuaca dan suhu terbukti memiliki pengaruh signifikan terhadap tingkat penyewaan sepeda. Penyewaan tertinggi terjadi ketika cuaca berada dalam kondisi cerah (Clear) dan suhu berada pada kategori hangat atau panas (Mild hingga Hot). Kondisi lingkungan yang nyaman tersebut mendorong masyarakat untuk lebih aktif menggunakan sepeda.

    Sebaliknya, ketika kondisi cuaca memburuk (Bad Weather), jumlah penyewaan menurun secara drastis meskipun suhu relatif hangat. Hal ini menunjukkan bahwa kondisi cuaca memiliki pengaruh yang lebih dominan dibandingkan suhu dalam menentukan keputusan pengguna untuk menyewa sepeda.

    Dengan demikian, dapat disimpulkan bahwa cuaca berperan sebagai faktor utama yang menentukan permintaan penyewaan sepeda, sedangkan suhu berfungsi sebagai faktor pendukung yang dapat memperkuat atau melemahkan tingkat permintaan ketika kondisi cuaca mendukung.